### 完全照抄原仓库renderer的变换逻辑

In [ ]:
import h5py
import numpy as np
import k3d
import trimesh
import os
from omegaconf import OmegaConf

import sys

pkg_root = os.path.abspath("airexo")
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)
# 如果之前在同一 kernel 里执行过 `from airexo.airexo...`，
# 需要清理缓存，避免命名空间包 `airexo` 残留。
for _m in [m for m in list(sys.modules.keys()) if m == "airexo" or m.startswith("airexo.")]:
    del sys.modules[_m]

from airexo.helpers.urdf_robot import forward_kinematic
from airexo.helpers.constants import ROBOT_PREDEFINED_TRANSFORMATION, O3D_RENDER_TRANSFORMATION
from airexo.calibration.calib_info import CalibrationInfo
import open3d as o3d

# ========== 1. 读取关节数据 ==========
scene_path = "/data/haoxiang/data/airexo2/task_0013/train/scene_0001"
lowdim_path = os.path.join(scene_path, "lowdim")

with h5py.File(f"{lowdim_path}/robot_left.h5", 'r') as f:
    left_joint = f['joint_pos'][0]
with h5py.File(f"{lowdim_path}/robot_right.h5", 'r') as f:
    right_joint = f['joint_pos'][0]
with h5py.File(f"{lowdim_path}/gripper_left.h5", 'r') as f:
    left_gripper = f['width'][0] if 'width' in f else 0.05
with h5py.File(f"{lowdim_path}/gripper_right.h5", 'r') as f:
    right_gripper = f['width'][0] if 'width' in f else 0.05

left_joint = np.concatenate([left_joint, [left_gripper]])
right_joint = np.concatenate([right_joint, [right_gripper]])

# ========== 2. 加载标定数据 ==========
calib_path = "/data/haoxiang/data/airexo2/task_0013/calib"
calib_info = CalibrationInfo(calib_path=calib_path, calib_timestamp=1737548651048)

camera_serial = list(calib_info.camera_serials_global)[0]
print(f"Using camera: {camera_serial}")

cam_to_base = calib_info.get_camera_to_base(camera_serial)
intrinsic = calib_info.get_intrinsic(camera_serial)

# ========== 3. 前向运动学 ==========
left_cfg = OmegaConf.load("airexo/airexo/configs/joint/left/robot.yaml")
right_cfg = OmegaConf.load("airexo/airexo/configs/joint/right/robot.yaml")

cur_transforms, visuals_map = forward_kinematic(
    left_joint=left_joint, right_joint=right_joint,
    left_joint_cfgs=left_cfg, right_joint_cfgs=right_cfg,
    is_rad=True,
    urdf_file="airexo/airexo/urdf_models/robot/robot_inhand.urdf",
    with_visuals_map=True
)

# ========== 4. 读取深度图并转换为点云 ==========
cam_path = os.path.join(scene_path, f"cam_{camera_serial}")
depth_path = os.path.join(cam_path, "depth", "1737546126606.png")
rgb_path = os.path.join(cam_path, "color", "1737546126606.png")

depth_img = o3d.io.read_image(depth_path)
rgb_img = o3d.io.read_image(rgb_path)
rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    rgb_img, depth_img, depth_scale=1000.0, convert_rgb_to_intensity=False
)

fx, fy = intrinsic[0,0], intrinsic[1,1]
cx, cy = intrinsic[0,2], intrinsic[1,2]
h, w = np.asarray(rgb_img).shape[:2]
intrinsic_o3d = o3d.camera.PinholeCameraIntrinsic(int(w), int(h), float(fx), float(fy), float(cx), float(cy))

pcd_cam = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic_o3d)
pcd_cam.transform(O3D_RENDER_TRANSFORMATION)

# ========== 5. K3D 可视化 ==========
plot = k3d.plot()

# 变换链：mesh → link → URDF基座 --[ROBOT_PREDEFINED]--> 真实基座
#         --[cam_to_base]--> 相机坐标系 --[O3D_RENDER]--> O3D渲染空间
for link, transform in cur_transforms.items():
    if link not in visuals_map: continue
    for v in visuals_map[link]:
        if v.geom_param is None: continue
        mesh_path = os.path.join("airexo/airexo/urdf_models/robot", v.geom_param)
        if not os.path.exists(mesh_path): continue
        mesh = trimesh.load(mesh_path, force='mesh')
        tf = O3D_RENDER_TRANSFORMATION @ cam_to_base @ ROBOT_PREDEFINED_TRANSFORMATION @ transform.matrix() @ v.offset.matrix()
        mesh.apply_transform(tf)
        plot += k3d.mesh(mesh.vertices.astype(np.float32), mesh.faces.astype(np.uint32), color=0xaaaaaa)

pcd_points = np.asarray(pcd_cam.points).astype(np.float32)
pcd_colors = np.asarray(pcd_cam.colors).astype(np.float32)
pcd_colors_uint32 = (pcd_colors * 255).astype(np.uint32)
pcd_colors_packed = (pcd_colors_uint32[:,0] << 16) | (pcd_colors_uint32[:,1] << 8) | pcd_colors_uint32[:,2]
plot += k3d.points(pcd_points, colors=pcd_colors_packed, point_size=0.002, shader='flat')

axis_size = 0.3
plot += k3d.vectors(
    origins=[[0,0,0],[0,0,0],[0,0,0]],
    vectors=[[axis_size,0,0],[0,axis_size,0],[0,0,axis_size]],
    colors=[0xff0000, 0x00ff00, 0x0000ff], line_width=0.01
)
plot.display()

### 直接用renderer（arm only，不含 gripper）

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import sys
from omegaconf import OmegaConf

pkg_root = os.path.abspath("airexo")
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)
for _m in [m for m in list(sys.modules.keys()) if m == "airexo" or m.startswith("airexo.")]:
    del sys.modules[_m]

from airexo.calibration.calib_info import CalibrationInfo
# ArmOnlyRobotRenderer 继承自 airexo RobotRenderer，渲染时过滤掉 gripper link，
# 使 render_mask() 只覆盖机械臂连杆（link1..7），不含夹爪。
from mask.renderer import ArmOnlyRobotRenderer

# ========== 1. 读取关节数据 ==========
scene_path = "/data/haoxiang/data/airexo2/task_0013/train/scene_0001"
lowdim_path = os.path.join(scene_path, "lowdim")

with h5py.File(f"{lowdim_path}/robot_left.h5", 'r') as f:
    left_joint = f['joint_pos'][0]

with h5py.File(f"{lowdim_path}/robot_right.h5", 'r') as f:
    right_joint = f['joint_pos'][0]

with h5py.File(f"{lowdim_path}/gripper_left.h5", 'r') as f:
    left_gripper = f['width'][0] if 'width' in f else 0.05

with h5py.File(f"{lowdim_path}/gripper_right.h5", 'r') as f:
    right_gripper = f['width'][0] if 'width' in f else 0.05

left_joint = np.concatenate([left_joint, [left_gripper]])
right_joint = np.concatenate([right_joint, [right_gripper]])

# ========== 2. 加载标定数据 ==========
calib_path = "/data/haoxiang/data/airexo2/task_0013/calib"
calib_info = CalibrationInfo(
    calib_path=calib_path,
    calib_timestamp=1737548651048
)

camera_serial = list(calib_info.camera_serials_global)[0]
print(f"Using camera: {camera_serial}")

cam_to_base = calib_info.get_camera_to_base(camera_serial)
intrinsic = calib_info.get_intrinsic(camera_serial)

# ========== 3. 加载配置 ==========
left_cfg  = OmegaConf.load("airexo/airexo/configs/joint/left/robot.yaml")
right_cfg = OmegaConf.load("airexo/airexo/configs/joint/right/robot.yaml")

# ========== 4. 初始化 Renderer（arm only） ==========
renderer = ArmOnlyRobotRenderer(
    left_joint_cfgs  = left_cfg,
    right_joint_cfgs = right_cfg,
    cam_to_base      = cam_to_base,
    intrinsic        = intrinsic,
    urdf_file        = "airexo/airexo/urdf_models/robot/robot_inhand.urdf",
    width=1280,
    height=720,
    near_plane=0.01,
    far_plane=100.0
)

# ========== 5. 更新关节并渲染 ==========
renderer.update_joints(left_joint, right_joint)

rendered_rgb   = renderer.render_image()
rendered_depth = renderer.render_depth()
rendered_mask  = renderer.render_mask(depth=rendered_depth)

# ========== 6. 读取真实相机图像 ==========
cam_path  = os.path.join(scene_path, f"cam_{camera_serial}")
rgb_path  = os.path.join(cam_path, "color", "1737546126606.png")
depth_path = os.path.join(cam_path, "depth", "1737546126606.png")

real_rgb   = cv2.cvtColor(cv2.imread(rgb_path), cv2.COLOR_BGR2RGB)
real_depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 1000.0

# ========== 7. 叠加显示（机械臂覆盖在真实图像上） ==========
alpha = 0.6
overlay_rgb = real_rgb.copy()
mask_bool = rendered_mask > 0
overlay_rgb[mask_bool] = (
    rendered_rgb[mask_bool] * alpha +
    real_rgb[mask_bool] * (1 - alpha)
).astype(np.uint8)

# ========== 8. 可视化 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(real_rgb);        axes[0, 0].set_title("Real RGB");              axes[0, 0].axis('off')
axes[0, 1].imshow(real_depth, cmap='plasma'); axes[0, 1].set_title("Real Depth"); axes[0, 1].axis('off')
axes[0, 2].imshow(overlay_rgb);     axes[0, 2].set_title("Overlay (arm only)");   axes[0, 2].axis('off')

axes[1, 0].imshow(rendered_rgb);    axes[1, 0].set_title("Rendered RGB");          axes[1, 0].axis('off')
axes[1, 1].imshow(rendered_depth, cmap='plasma'); axes[1, 1].set_title("Rendered Depth"); axes[1, 1].axis('off')
axes[1, 2].imshow(rendered_mask, cmap='gray'); axes[1, 2].set_title("Mask (no gripper)"); axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

# ========== 9. 保存结果 ==========
output_dir = os.path.join(scene_path, "rendered_output")
os.makedirs(output_dir, exist_ok=True)

cv2.imwrite(os.path.join(output_dir, "rendered_rgb.png"),   cv2.cvtColor(rendered_rgb, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, "rendered_depth.png"), (rendered_depth * 1000).astype(np.uint16))
cv2.imwrite(os.path.join(output_dir, "rendered_mask.png"),  rendered_mask)
cv2.imwrite(os.path.join(output_dir, "overlay.png"),        cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))

print(f"\n结果已保存到: {output_dir}")
print(f"渲染图像形状: {rendered_rgb.shape}")
print(f"深度范围: [{rendered_depth.min():.3f}, {rendered_depth.max():.3f}] 米")
print(f"掩码覆盖像素: {(rendered_mask > 0).sum()} / {rendered_mask.size}")